# Verify CruiseKube recommendations vs cluster

This notebook checks three things:

1. **Workload table vs cluster** — `workloads.workload_id` should match supported cluster workloads (Deployments, StatefulSets, DaemonSets with selectors), the same set CreateStats maintains.
2. **Recommendation table vs cluster** — `pod_resource_recommendations.workload_id` should match the workload implied by each pod’s labels, and recommendation JSON CPU/memory should match live pod requests/limits (existing check).
3. **Resources in workloads.stats** — `original_container_resources` in the stored stats JSON should match the workload’s current pod template in the API.


In [6]:
# Install dependencies if needed (uncomment in a fresh env)
# %pip install -q -r requirements.txt
# %pip install pandas




In [1]:
import os
from pathlib import Path

# Optional: set DB via env or use config file
# os.environ["DB_HOST"] = "localhost"
# os.environ["DB_PORT"] = "5432"
# os.environ["DB_NAME"] = "cruisekube"
# os.environ["DB_USER"] = "cruisekube"
# os.environ["DB_PASSWORD"] = "cruisekube"


CLUSTER_ID = os.environ.get("CLUSTER_ID", "default")  # change to your cluster ID
CONFIG_PATH = Path("config.yaml")  # path to config.yaml with db section (or None to use env)
KUBECONFIG = os.environ.get("KUBECONFIG", "")  # or path to kubeconfig

In [3]:
from verify_recommendations import (
    run_verification,
    run_workload_db_cluster_alignment,
    summarize_alignment,
)

rows, mismatches = run_verification(
    cluster_id=CLUSTER_ID,
    config_path=str(CONFIG_PATH) if CONFIG_PATH.exists() else None,
    kubeconfig_path=KUBECONFIG or None,
)
print(f"Fetched {len(rows)} recommendations for cluster {CLUSTER_ID}")
print(f"Mismatches (or errors): {len(mismatches)}")

align_report = run_workload_db_cluster_alignment(
    cluster_id=CLUSTER_ID,
    rows=rows,
    config_path=str(CONFIG_PATH) if CONFIG_PATH.exists() else None,
    kubeconfig_path=KUBECONFIG or None,
)
print("\n--- DB / cluster workload alignment ---\n")
print(summarize_alignment(align_report))


Fetched 882 recommendations for cluster default
Mismatches (or errors): 662


In [4]:
import json
from dataclasses import asdict, dataclass

path = Path("mismatch.json")
data = [asdict(m) for m in mismatches]
with open(path, "w") as f:
    json.dump(data, f, indent=2)
print(f"Saved {len(mismatches)} mismatches to {path}")

Saved 662 mismatches to mismatch.json


## Load from mismatch.json

Load previously saved results from `mismatch.json` and display in a pandas table (run this after a verification run has saved the file, or in a new session).

In [9]:
import pandas as pd

data = []
path = Path("mismatch.json")
if not path.exists():
    data = []
else:
    with open(path) as f:
        data = json.load(f)


if data:
    df = pd.DataFrame(data)
    print(f"Loaded {len(df)} mismatches from mismatch.json")
    df
else:
    print("No mismatch.json found or file is empty. Run the verification cells first to save results.")

Loaded 662 mismatches from mismatch.json


In [10]:
# Optional: summary as a table (pandas)
try:
    import pandas as pd
    data = []
    for m in mismatches:
        data.append({
            "namespace": m.namespace,
            "pod": m.pod,
            "container": m.container,
            "workload_id": m.workload_id,
            "error": m.error or "-",
            "cpu_req_diff": m.cpu_request_diff,
            "mem_req_diff": m.memory_request_diff,
            "cpu_lim_diff": m.cpu_limit_diff,
            "mem_lim_diff": m.memory_limit_diff,
            "recommended_cpu_request": m.recommended_cpu_request,
            "actual_cpu_request": m.actual_cpu_request,
            "recommended_memory_request": m.recommended_memory_request,
            "actual_memory_request": m.actual_memory_request,
            "cpu_diff": m.actual_cpu_request - m.recommended_cpu_request,
            "mem_diff": m.actual_memory_request - m.recommended_memory_request,
            # "recommended_cpu_limit": m.recommended_cpu_limit,
            # "actual_cpu_limit": m.actual_cpu_limit,
            # "recommended_memory_limit": m.recommended_memory_limit,
            # "actual_memory_limit": m.actual_memory_limit,
        })
    if data:
        df = pd.DataFrame(data)
        df
    else:
        print("No mismatches to display.")
except ImportError:
    print("Install pandas for DataFrame view: pip install pandas")

In [11]:
# Only rows where CPU request or Memory request differ (ignore limit differences)
req_mask = df["cpu_req_diff"] | df["mem_req_diff"]
df_request = df.loc[req_mask, ["workload_id", "pod", "container", "recommended_cpu_request", "actual_cpu_request", "cpu_diff", "recommended_memory_request", "actual_memory_request",  "mem_diff"]].copy()
df_request = df_request.rename(columns={
    "workload_id": "workload",
    "recommended_cpu_request": "cpu_rec",
    "actual_cpu_request": "cpu_act",
    "cpu_diff": "cpu_diff",
    "recommended_memory_request": "mem_rec_MB",
    "actual_memory_request": "mem_act_MB",
    "mem_diff": "mem_diff",
})
df_request.to_excel("mismatch_request.xlsx", index=False)

ModuleNotFoundError: No module named 'openpyxl'

In [12]:
# Print all rows 
pd.set_option('display.max_rows', 500)
df_request.sort_values("cpu_diff", ascending=False)

,workload,pod,container,cpu_rec,cpu_act,cpu_diff,mem_rec_MB,mem_act_MB,mem_diff
369,Deployment:truefoundry:deltafusion-query-server,deltafusion-query-server-99bc775ff-hwjk8,deltafusion-query-server,0.957,3.000,2.043,1224.0,12000.000000,10776.000000
0,Deployment:truefoundry:deltafusion-query-server,deltafusion-query-server-99bc775ff-jtbnd,deltafusion-query-server,1.112,3.000,1.888,1640.0,12000.000000,10360.000000
80,StatefulSet:vivek-ks-devtest:test-spark-cluste...,test-spark-cluster-worker-1,spark-worker,0.681,2.000,1.319,1743.0,4294.967296,2551.967296
376,StatefulSet:vivek-ks-devtest:test-spark-cluste...,test-spark-cluster-worker-3,spark-worker,0.967,2.000,1.033,1743.0,4294.967296,2551.967296
156,Deployment:jump-trading:truefoundry-deltafusio...,truefoundry-deltafusion-query-server-847bf9666...,deltafusion-query-server,0.001,1.000,0.999,18.0,4000.000000,3982.000000
155,Deployment:multi-tenant-test:truefoundry-delta...,truefoundry-deltafusion-query-server-5b5d69b66...,deltafusion-query-server,0.002,1.000,0.998,50.0,4000.000000,3950.000000
3,StatefulSet:vivek-ks-devtest:test-spark-cluste...,test-spark-cluster-master-0,spark-master,0.007,1.000,0.993,416.0,2147.483648,1731.483648
539,StatefulSet:vivek-ks-devtest:test-spark-cluste...,test-spark-cluster-worker-4,spark-worker,1.062,2.000,0.938,1743.0,4294.967296,2551.967296
1,StatefulSet:vivek-ks-devtest:test-spark-cluste...,test-spark-cluster-worker-2,spark-worker,1.118,2.000,0.882,1743.0,4294.967296,2551.967296
178,Deployment:truefoundry:tfy-k8s-controller,tfy-k8s-controller-58f67b87d6-csqrk,tfy-k8s-controller,0.090,0.750,0.660,812.0,2400.000000,1588.000000


In [55]:
sum(df_request["cpu_diff"])

1.2480000000000004

In [56]:
df_request[df_request["pod"]=="kube-proxy-n4ww4"]

,workload,pod,container,cpu_rec,cpu_act,cpu_diff,mem_rec_MB,mem_act_MB,mem_diff
723,DaemonSet:kube-system:kube-proxy,kube-proxy-n4ww4,kube-proxy,0.0,0.0,0.0,16.0,0.0,-16.0


In [57]:
# Get unique workloads
workloads = df_request["workload"].unique()
print(len(workloads))
print(workloads[1])



92
DaemonSet:kube-system:aws-node
